In [0]:
import uuid, re
from datetime import datetime, timezone
from pyspark.sql import functions as F

CATALOG, Y = "workspace", 2025
BRONZE = f"{CATALOG}.rc_bronze.survey_results"
SILVER = f"{CATALOG}.rc_silver.survey_clean"
MULTI = f"{CATALOG}.rc_silver.survey_multi"
ANSWERED = f"{CATALOG}.rc_silver.survey_multi_answered"
INGEST_LOG = f"{CATALOG}.rc_bronze.ingestion_log"
SCHEMA_CSV = f"/Volumes/{CATALOG}/ai_reality_check/raw/survey_{Y}/schema.csv"

# Seuils d'alerte : choix d'analyste, modifiables ici
WARN_NULL_RATE = 0.60
WARN_FLAG_RATE = 0.05

RUN_ID = str(uuid.uuid4())
STARTED = datetime.now(timezone.utc).replace(tzinfo=None)
results = []

def check(layer, name, severity, passed, observed, expected, detail=""):
    status = "PASS" if passed else "FAIL"
    results.append((RUN_ID, layer, name, severity, status,
                    str(observed)[:200], str(expected)[:200], str(detail)[:300]))
    print(f"[{status}] {severity:<8} {layer:<6} {name} | observé : {observed} | attendu : {expected}")

bz = spark.table(BRONZE).where(f"_survey_year = {Y}")
sv = spark.table(SILVER).where(f"_survey_year = {Y}")
mu = spark.table(MULTI).where(f"survey_year = {Y}")
an = spark.table(ANSWERED).where(f"survey_year = {Y}")
src_cols = [r.bronze_name for r in spark.table(f"{CATALOG}.rc_bronze.column_map")
            .where(f"survey_year = {Y}").orderBy("ordinal").collect()]
n_b, n_s = bz.count(), sv.count()

# --- BRONZE : schéma ---
sch = (spark.read.option("header", True).option("multiLine", True)
       .option("quote", '"').option("escape", '"').csv(SCHEMA_CSV))
san = lambda x: re.sub(r"[^0-9a-zA-Z]+", "_", x).strip("_")
q_s = [san(r.qname) for r in sch.select("qname").distinct().collect()]
unmapped_q = [q for q in q_s if not any(c == q or c.startswith(q) for c in src_cols)]
unmapped_c = [c for c in src_cols if not any(c == q or c.startswith(q) for q in q_s)]
check("bronze", "nombre_de_colonnes_source", "critical", len(src_cols) == 172, len(src_cols), 172)
check("bronze", "chaque_question_du_schema_a_une_colonne", "critical", not unmapped_q, len(unmapped_q), 0, unmapped_q)
check("bronze", "colonnes_sans_entree_de_schema", "warning",
      set(unmapped_c) <= {"ResponseId", "ConvertedCompYearly"}, sorted(unmapped_c), "ResponseId, ConvertedCompYearly")

# --- BRONZE : clé et journal d'ingestion ---
k = bz.agg(F.countDistinct("ResponseId").alias("d"),
           F.sum(F.col("ResponseId").isNull().cast("int")).alias("nul")).collect()[0]
check("bronze", "responseid_unique_et_non_nul", "critical",
      k["d"] == n_b and k["nul"] == 0, f"{k['d']}/{n_b}, nuls={k['nul']}", "distincts = lignes, 0 nul")
last = (spark.table(INGEST_LOG).where(f"survey_year = {Y}")
        .orderBy(F.desc("started_at")).limit(1).collect()[0])
check("bronze", "dernier_ingest_success_et_lignes_ecrites", "critical",
      last["status"] == "SUCCESS" and last["rows_written"] == n_b,
      f"{last['status']}, écrites={last['rows_written']}, table={n_b}", "SUCCESS et égalité")

[PASS] critical bronze nombre_de_colonnes_source | observé : 172 | attendu : 172
[PASS] critical bronze chaque_question_du_schema_a_une_colonne | observé : 0 | attendu : 0
[PASS] warning  bronze colonnes_sans_entree_de_schema | observé : ['ConvertedCompYearly', 'ResponseId'] | attendu : ResponseId, ConvertedCompYearly
[PASS] critical bronze responseid_unique_et_non_nul | observé : 49191/49191, nuls=0 | attendu : distincts = lignes, 0 nul
[PASS] critical bronze dernier_ingest_success_et_lignes_ecrites | observé : SUCCESS, écrites=49191, table=49191 | attendu : SUCCESS et égalité


In [0]:
# --- Réconciliation Bronze -> Silver, colonne par colonne ---
bz_na = bz.agg(*[F.sum((F.col(c) == "NA").cast("int")).alias(c) for c in src_cols]).collect()[0].asDict()
sv_nu = sv.agg(*[F.sum(F.col(c).isNull().cast("int")).alias(c) for c in src_cols]).collect()[0].asDict()
bad = [c for c in src_cols if (bz_na[c] or 0) != (sv_nu[c] or 0)]
check("silver", "lignes_bronze_egal_silver", "critical", n_s == n_b, f"{n_s} vs {n_b}", "égalité")
check("silver", "manquants_NA_bronze_egal_NULL_silver", "critical", not bad, len(bad), 0, bad)

str_cols = [c for c, t in sv.dtypes if t == "string" and c in src_cols]
na_left = sv.agg(*[F.sum((F.col(c) == "NA").cast("int")).alias(c) for c in str_cols]).collect()[0].asDict()
left = sum((v or 0) for v in na_left.values())
check("silver", "aucune_chaine_NA_restante", "critical", left == 0, left, 0)

d_s = sv.agg(F.countDistinct("ResponseId").alias("d")).collect()[0]["d"]
check("silver", "responseid_unique_et_non_nul", "critical",
      d_s == n_s and sv_nu["ResponseId"] == 0, f"{d_s}/{n_s}", "distincts = lignes")

# --- Taux de nuls ---
for c in ["MainBranch", "Age"]:
    check("silver", f"{c}_toujours_renseigne", "critical", sv_nu[c] == 0, sv_nu[c], 0)
for c in ["AISelect", "AISent", "AIAcc", "JobSat", "ConvertedCompYearly"]:
    rate = sv_nu[c] / n_s
    check("silver", f"taux_de_nuls_{c}", "warning", rate <= WARN_NULL_RATE, f"{rate:.1%}", f"<= {WARN_NULL_RATE:.0%}")

# --- Plages numériques ---
r = sv.agg(F.min("WorkExp").alias("we_min"), F.max("WorkExp").alias("we_max"),
           F.min("YearsCode").alias("yc_min"), F.max("YearsCode").alias("yc_max"),
           F.min("JobSat").alias("js_min"), F.max("JobSat").alias("js_max"),
           F.min("ConvertedCompYearly").alias("comp_min")).collect()[0]
check("silver", "plage_WorkExp_0_100", "critical", 0 <= r["we_min"] and r["we_max"] <= 100, f"{r['we_min']}..{r['we_max']}", "0..100")
check("silver", "plage_YearsCode_0_100", "critical", 0 <= r["yc_min"] and r["yc_max"] <= 100, f"{r['yc_min']}..{r['yc_max']}", "0..100")
check("silver", "plage_JobSat_0_10", "critical", 0 <= r["js_min"] and r["js_max"] <= 10, f"{r['js_min']}..{r['js_max']}", "0..10")
check("silver", "salaire_strictement_positif", "critical", r["comp_min"] > 0, r["comp_min"], "> 0")

both = sv.where("WorkExp is not null and YearsCode is not null")
inc = both.where("WorkExp > YearsCode").count()
check("silver", "WorkExp_inferieur_ou_egal_YearsCode", "warning", inc / both.count() <= WARN_FLAG_RATE,
      f"{inc} lignes ({inc / both.count():.1%})", f"<= {WARN_FLAG_RATE:.0%}")

ex = sv.agg(F.sum(F.col("is_salary_extreme").cast("int")).alias("e"),
            F.count("ConvertedCompYearly").alias("n")).collect()[0]
check("silver", "taux_de_salaires_extremes", "warning", ex["e"] / ex["n"] <= WARN_FLAG_RATE,
      f"{ex['e']} ({ex['e'] / ex['n']:.1%})", f"<= {WARN_FLAG_RATE:.0%}")

# --- Doublons et lignes quasi vides ---
low = sv.where("is_low_effort").count()
dup = sv.where("is_content_duplicate").count()
dup_big = sv.where("is_content_duplicate and n_answered > 20").count()
check("silver", "taux_lignes_quasi_vides", "warning", low / n_s <= WARN_FLAG_RATE, f"{low} ({low / n_s:.1%})", f"<= {WARN_FLAG_RATE:.0%}")
check("silver", "taux_doublons_de_contenu", "warning", dup / n_s <= WARN_FLAG_RATE, f"{dup} ({dup / n_s:.1%})", f"<= {WARN_FLAG_RATE:.0%}")
check("silver", "aucun_doublon_au_dela_de_20_reponses", "warning", dup_big == 0, dup_big, 0)

# --- Catégories attendues (libellés relevés à l'EDA) ---
EXPECTED = {
    "AISelect": ["Yes, I use AI tools daily", "Yes, I use AI tools weekly",
                 "Yes, I use AI tools monthly or infrequently", "No, but I plan to soon", "No, and I don't plan to"],
    "AISent": ["Very favorable", "Favorable", "Indifferent", "Unsure", "Unfavorable", "Very unfavorable"],
    "AIAcc": ["Highly trust", "Somewhat trust", "Neither trust nor distrust", "Somewhat distrust", "Highly distrust"],
    "MainBranch": ["I am a developer by profession", "I am learning to code",
                   "I am not primarily a developer, but I write code sometimes as part of my work/studies",
                   "I code primarily as a hobby",
                   "I work with developers or my work supports developers but am not a developer by profession",
                   "I used to be a developer by profession, but no longer am"],
    "ICorPM": ["Individual contributor", "People manager"],
    "AIThreat": ["Yes", "No", "I'm not sure"],
    "Age": ["18-24 years old", "25-34 years old", "35-44 years old", "45-54 years old",
            "55-64 years old", "65 years or older", "Prefer not to say"],
}
CRITICAL_CATS = {"AISelect", "AISent", "AIAcc", "MainBranch"}
for col, labels in EXPECTED.items():
    unexpected = (sv.where(F.col(col).isNotNull() & ~F.col(col).isin(labels))
                    .groupBy(col).count().collect())
    check("silver", f"categories_valides_{col}", "critical" if col in CRITICAL_CATS else "warning",
          not unexpected, len(unexpected), 0, [(u[0], u[1]) for u in unexpected][:5])

[PASS] critical silver lignes_bronze_egal_silver | observé : 49191 vs 49191 | attendu : égalité
[PASS] critical silver manquants_NA_bronze_egal_NULL_silver | observé : 0 | attendu : 0
[PASS] critical silver aucune_chaine_NA_restante | observé : 0 | attendu : 0
[PASS] critical silver responseid_unique_et_non_nul | observé : 49191/49191 | attendu : distincts = lignes
[PASS] critical silver MainBranch_toujours_renseigne | observé : 0 | attendu : 0
[PASS] critical silver Age_toujours_renseigne | observé : 0 | attendu : 0
[PASS] warning  silver taux_de_nuls_AISelect | observé : 31.5% | attendu : <= 60%
[PASS] warning  silver taux_de_nuls_AISent | observé : 32.0% | attendu : <= 60%
[PASS] warning  silver taux_de_nuls_AIAcc | observé : 32.3% | attendu : <= 60%
[PASS] warning  silver taux_de_nuls_JobSat | observé : 45.8% | attendu : <= 60%
[PASS] warning  silver taux_de_nuls_ConvertedCompYearly | observé : 51.3% | attendu : <= 60%
[PASS] critical silver plage_WorkExp_0_100 | observé : 1..100 |

In [0]:
# --- Réponses multiples : double comptage et intégrité ---
dups = mu.groupBy("ResponseId", "source_column", "item_norm").count().where("count > 1").count()
check("multi", "aucun_double_comptage", "critical", dups == 0, dups, 0)
orph = mu.select("ResponseId").distinct().join(sv.select("ResponseId"), "ResponseId", "left_anti").count()
orph2 = an.select("ResponseId").distinct().join(sv.select("ResponseId"), "ResponseId", "left_anti").count()
check("multi", "integrite_referentielle_vers_survey_clean", "critical", orph == 0 and orph2 == 0, f"{orph}/{orph2}", "0/0")
empty = mu.where("item = ''").count()
check("multi", "aucun_element_vide", "critical", empty == 0, empty, 0)

KEY_GROUPS = {"LearnCode": ["LearnCode"], "AIFrustration": ["AIFrustration"],
              "AITool": [c for c in src_cols if c.startswith("AITool")],
              "AIAgentImpact": [c for c in src_cols if c.startswith("AIAgentImpact")],
              "AIAgentChallenges": [c for c in src_cols if c.startswith("AIAgentChallenges")]}
from functools import reduce
got = {x["question_group"]: x["count"] for x in an.groupBy("question_group").count().collect()}
for g, cs in KEY_GROUPS.items():
    cond = reduce(lambda a, b: a | b, [F.col(c).isNotNull() for c in cs])
    e = sv.where(cond).count()
    check("multi", f"repondants_groupe_{g}", "critical", e == got.get(g), f"{got.get(g)} vs {e}", "égalité")

# --- Enregistrement et statut ---
schema = ("run_id string, layer string, check_name string, severity string, status string, "
          "observed string, expected string, detail string")
(spark.createDataFrame(results, schema).withColumn("checked_at", F.current_timestamp())
      .write.format("delta").mode("append").saveAsTable(f"{CATALOG}.rc_silver.dq_results"))

n_crit = sum(1 for x in results if x[3] == "critical" and x[4] == "FAIL")
n_warn = sum(1 for x in results if x[3] == "warning" and x[4] == "FAIL")
status = "FAILED" if n_crit else ("SUCCESS_WITH_WARNINGS" if n_warn else "SUCCESS")
run = [(RUN_ID, Y, len(results), n_crit, n_warn, status, STARTED, datetime.now(timezone.utc).replace(tzinfo=None))]
(spark.createDataFrame(run, "run_id string, survey_year int, n_checks int, n_critical_failed int, "
                            "n_warning_failed int, status string, started_at timestamp, finished_at timestamp")
      .write.format("delta").mode("append").saveAsTable(f"{CATALOG}.rc_silver.dq_runs"))

print(f"\nSTATUT : {status} | {len(results)} contrôles | échecs critiques : {n_crit} | avertissements : {n_warn}")
display(spark.table(f"{CATALOG}.rc_silver.dq_results").where(f"run_id = '{RUN_ID}'")
        .where("status = 'FAIL'").select("layer", "check_name", "severity", "observed", "expected", "detail"))
if n_crit:
    raise Exception(f"{n_crit} contrôle(s) critique(s) en échec, voir rc_silver.dq_results (run {RUN_ID})")

[PASS] critical multi  aucun_double_comptage | observé : 0 | attendu : 0
[PASS] critical multi  integrite_referentielle_vers_survey_clean | observé : 0/0 | attendu : 0/0
[PASS] critical multi  aucun_element_vide | observé : 0 | attendu : 0
[PASS] critical multi  repondants_groupe_LearnCode | observé : 33556 vs 33556 | attendu : égalité
[PASS] critical multi  repondants_groupe_AIFrustration | observé : 31522 vs 31522 | attendu : égalité
[PASS] critical multi  repondants_groupe_AITool | observé : 31066 vs 31066 | attendu : égalité
[PASS] critical multi  repondants_groupe_AIAgentImpact | observé : 12840 vs 12840 | attendu : égalité
[PASS] critical multi  repondants_groupe_AIAgentChallenges | observé : 28954 vs 28954 | attendu : égalité

STATUT : SUCCESS_WITH_WARNINGS | 40 contrôles | échecs critiques : 0 | avertissements : 1


layer,check_name,severity,observed,expected,detail
silver,WorkExp_inferieur_ou_egal_YearsCode,warning,3564 lignes (8.9%),<= 5%,


In [0]:
from pyspark.sql import functions as F

CATALOG, Y = "workspace", 2025
sv = spark.table(f"{CATALOG}.rc_silver.survey_clean").where(f"_survey_year = {Y}")

print("=== Libellés officiels (schema.csv) ===")
sch = (spark.read.option("header", True).option("multiLine", True)
       .option("quote", '"').option("escape", '"')
       .csv(f"/Volumes/{CATALOG}/ai_reality_check/raw/survey_{Y}/schema.csv"))
sch.where(F.col("qname").isin("WorkExp", "YearsCode", "JobSat")) \
   .select("qname", "question", "type").show(truncate=False)

both = (sv.where("WorkExp is not null and YearsCode is not null")
          .withColumn("gap", F.col("WorkExp") - F.col("YearsCode")))
n = both.count()
print(f"=== Répondants avec les deux valeurs : {n:,} ===")
(both.withColumn("tranche",
        F.when(F.col("gap") < 0, "a) WorkExp < YearsCode")
         .when(F.col("gap") == 0, "b) égaux")
         .when(F.col("gap") <= 2, "c) WorkExp > YearsCode de 1 à 2 ans")
         .when(F.col("gap") <= 10, "d) WorkExp > YearsCode de 3 à 10 ans")
         .otherwise("e) WorkExp > YearsCode de plus de 10 ans"))
     .groupBy("tranche").count()
     .withColumn("pct", F.round(F.col("count") / n * 100, 1))
     .orderBy("tranche").show(truncate=False))

print("=== Écart positif, par profil ===")
(both.groupBy("is_professional")
     .agg(F.count("*").alias("n"), F.sum((F.col("gap") > 0).cast("int")).alias("n_ecart_positif"))
     .withColumn("pct", F.round(F.col("n_ecart_positif") / F.col("n") * 100, 1)).show())

=== Libellés officiels (schema.csv) ===
+---------+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+----+
|qname    |question                                                                                                                                                                       |type|
+---------+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+----+
|WorkExp  |How many years of professional work experience do you have? Please round to the nearest whole number, excluding any decimal points.  If your answer is '0', please leave blank.|TE  |
|YearsCode|Including any education, how many years have you been coding in total?  Please round to the nearest whole number, excluding any decimal points.                                |T